# Test B and S outputs

Checks ciphertext-derived B, trace-derived B consistency across xs traces, and S consistency across runs.

In [14]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

OUTPUT_DIR = Path("output_decapsulation_truncated")
B_DIR = OUTPUT_DIR / "B"
S_DIR = OUTPUT_DIR / "S"

PARAMS_N = 640
PARAMS_NBAR = 8
PARAMS_LOGQ = 15
CRYPTO_CIPHERTEXTBYTES = 9720
BYTES_CIPHERTEXT_C1 = (PARAMS_LOGQ * PARAMS_N * PARAMS_NBAR) // 8

S_OFFSET = 16 + 16 + 9600
S_LENGTH = PARAMS_N * PARAMS_NBAR * 2

def run_id(path):
    return int(re.search(r"_(\d+)", path.stem).group(1))

def unpack_c1(c1):
    values = []
    for i in range(PARAMS_NBAR):
        for j in range(PARAMS_N):
            start = (i * PARAMS_N + j) * PARAMS_LOGQ
            val = 0
            for bit in range(PARAMS_LOGQ):
                byte_pos = (start + bit) >> 3
                bit_pos = 7 - ((start + bit) & 7)
                val |= ((c1[byte_pos] >> bit_pos) & 1) << (PARAMS_LOGQ - 1 - bit)
            values.append(val)
    return np.array(values, dtype=np.uint16).reshape(PARAMS_NBAR, PARAMS_N)

def read_B_csv(path, dtype):
    cols = [f"B_col_{j}" for j in range(PARAMS_N)]
    return pd.read_csv(path)[cols].to_numpy(dtype=dtype)

# The method to read S from csv file 
def read_S_snapshot_csv(path):
    cols = [f"S_col_{j}" for j in range(PARAMS_NBAR)]
    return pd.read_csv(path)[cols].to_numpy(dtype=np.int16)

# The S that is computed by extracting S from registers
def read_S_register_csv(path):
    cols = [f"S_col_{j}" for j in range(PARAMS_NBAR)]
    return pd.read_csv(path)[cols].to_numpy(dtype=np.int16)

# The S that is computed by extracting S from the secret key file
def extract_S_from_sk(path):
    sk = path.read_bytes()
    S_bytes = sk[S_OFFSET:S_OFFSET + S_LENGTH]
    if len(S_bytes) != S_LENGTH:
        raise ValueError(f"S has wrong size: {len(S_bytes)} != {S_LENGTH}")
    return np.frombuffer(S_bytes, dtype="<i2").reshape(PARAMS_NBAR, PARAMS_N).T


In [15]:
ciphertext_paths = sorted(OUTPUT_DIR.glob("ciphertext_*.bin"), key=run_id)
runs = [run_id(path) for path in ciphertext_paths]

if not runs:
    raise FileNotFoundError(f"No ciphertext_*.bin files found in {OUTPUT_DIR}")

rows = []
for i, ct_path in zip(runs, ciphertext_paths):
    ct = ct_path.read_bytes()
    if len(ct) != CRYPTO_CIPHERTEXTBYTES:
        raise ValueError(f"{ct_path} has wrong size: {len(ct)}")

    # B from ciphertext 
    B_from_ct = unpack_c1(ct[:BYTES_CIPHERTEXT_C1])
    #B from the csv file that should be the same form B for ciphertext 
    B = read_B_csv(B_DIR / f"B_{i}.csv", np.uint16)
    # B from registers 
    B_registers= read_B_csv(B_DIR / f"B_from_registers_{i}.csv", np.uint16)

    rows.append({
        "run": i,
        "B_matches_ciphertext": bool(np.array_equal(B, B_from_ct)),
        "B_registers_unsigned_matches_B": bool(np.array_equal(B_registers, B)),
        "B_min": int(B.min()),
        "B_max": int(B.max()),
    })

B_results = pd.DataFrame(rows)
display(B_results)

# print if the values fon't match 
if not B_results["B_matches_ciphertext"].all():
    print("B does not match ciphertext for runs:", B_results[~B_results["B_matches_ciphertext"]]["run"].tolist())
else:
    print("All B values match the ciphertext")
assert B_results["B_matches_ciphertext"].all()
assert B_results["B_registers_unsigned_matches_B"].all()


,run,B_matches_ciphertext,B_registers_unsigned_matches_B,B_min,B_max
0,0,True,True,3,32758
1,1,True,True,0,32767
2,2,True,True,0,32746
3,3,True,True,14,32763
4,4,True,True,11,32766
...,...,...,...,...,...
75,75,True,True,14,32744
76,76,True,True,2,32757
77,77,True,True,0,32764
78,78,True,True,4,32767


All B values match the ciphertext


In [16]:
S_from_sk = extract_S_from_sk(OUTPUT_DIR / "sk.bin")

rows = []
S_reference = None

if (S_DIR / "S.csv").exists():
    S_snapshot = read_S_snapshot_csv(S_DIR / "S.csv")
    rows.append({
        "name": "S.csv from sk",
        "matches_sk": bool(np.array_equal(S_snapshot, S_from_sk)),
        "matches_S_0": None,
    })

for i in runs:
    S_i = read_S_register_csv(S_DIR / f"S_{i}.csv")
    if S_reference is None:
        S_reference = S_i

    rows.append({
        "name": f"S_{i}.csv from registers",
        "matches_sk": bool(np.array_equal(S_i, S_from_sk)),
        "matches_S_0": bool(np.array_equal(S_i, S_reference)),
    })

S_results = pd.DataFrame(rows)
display(S_results)

assert S_results["matches_sk"].all()
assert S_results[S_results["matches_S_0"].notna()]["matches_S_0"].all()

,name,matches_sk,matches_S_0
0,S.csv from sk,True,None
1,S_0.csv from registers,True,True
2,S_1.csv from registers,True,True
3,S_2.csv from registers,True,True
4,S_3.csv from registers,True,True
...,...,...,...
76,S_75.csv from registers,True,True
77,S_76.csv from registers,True,True
78,S_77.csv from registers,True,True
79,S_78.csv from registers,True,True
